# GridNav-AI Demo 🤖🗺️

This notebook demonstrates the **GridNav-AI** pathfinding model.

A ResNet-based CNN trained to navigate a robot from start `R` to target `T` on a randomly generated grid — while avoiding obstacles.

---

### What this notebook does:
1. Loads the pre-trained ResNet model
2. Generates a random unseen grid
3. Simulates the robot navigating to the target
4. Animates the robot's path live

> ⚠️ Make sure you have trained the model first by running `src/pathfinder.py`
> The trained model should be saved at `examples/path_prediction_mlp.pth`

## 1. Imports

In [5]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# Add src/ to path so we can import from pathfinder.py
sys.path.append(os.path.join(os.path.dirname(''), '../src'))

from pathfinder import (
    PathPredictionResNet,
    generate_random_grid,
    simulate_robot_movement,
    animate_path,
    ACTIONS
)

print('✅ Imports successful!')

✅ Imports successful!


## 2. Configuration

In [ ]:
# Grid configuration — must match what the model was trained on
MAP_ROWS = 20
MAP_COLS = 20
IN_CHANNELS = 3
OUTPUT_DIM = len(ACTIONS)  # 8 possible moves
OBSTACLE_DENSITY = 0.25
MODEL_PATH = '../models/path_prediction_mlp.pth'

print(f'Grid size: {MAP_ROWS}x{MAP_COLS}')
print(f'Possible actions: {OUTPUT_DIM}')
print(f'Model path: {MODEL_PATH}')

Grid size: 20x20
Possible actions: 8
Model path: ../examples/path_prediction_mlp.pth


## 3. Load Pre-trained Model

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Initialize model architecture
model = PathPredictionResNet(
    in_channels=IN_CHANNELS,
    height=MAP_ROWS,
    width=MAP_COLS,
    output_dim=OUTPUT_DIM
)

# Load trained weights
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.to(device)
    model.eval()
    print('✅ Model loaded successfully!')
else:
    print(f'❌ Model not found at {MODEL_PATH}')
    print('Please run src/pathfinder.py first to train the model.')

Using device: cpu
❌ Model not found at ../examples/path_prediction_mlp.pth
Please run src/pathfinder.py first to train the model.


## 4. Generate a Random Grid

In [ ]:
# Generate a random unseen grid
grid_data, numeric_grid, optimal_path, robot_pos, target_pos = generate_random_grid(
    MAP_ROWS, MAP_COLS, OBSTACLE_DENSITY
)

if grid_data is None:
    print('❌ Could not generate a solvable grid. Try again.')
else:
    print(f'✅ Grid generated successfully!')
    print(f'Robot start: {robot_pos}')
    print(f'Target position: {target_pos}')
    print(f'Optimal BFS path length: {len(optimal_path)} steps')

## 5. Visualize the Grid

In [ ]:
# Visualize the starting grid
colors = ['#FFFFFF', '#DDDDDD', '#000000', '#FF0000', '#00FF00']
cmap = plt.matplotlib.colors.ListedColormap(colors)
bounds = [0.0, 0.25, 0.75, 1.5, 2.5, 3.5]
norm = plt.matplotlib.colors.BoundaryNorm(bounds, cmap.N)

plt.figure(figsize=(6, 6))
plt.imshow(numeric_grid, cmap=cmap, norm=norm, origin='upper')
plt.title('Starting Grid\n🔴 Robot | 🟢 Target | ⬛ Obstacle')
plt.axis('off')
plt.tight_layout()
plt.show()

print('Legend: White=Empty, Black=Obstacle, Red=Robot, Green=Target')

## 6. Simulate Robot Navigation

In [ ]:
# Run the trained model on the grid
path_taken, frames = simulate_robot_movement(
    model,
    grid_data,
    max_steps=100,
    model_name='ResNet'
)

if path_taken:
    print(f'\n✅ Simulation complete!')
    print(f'Steps taken by AI: {len(path_taken) - 1}')
    print(f'Optimal BFS steps: {len(optimal_path) - 1}')
    efficiency = len(optimal_path) / len(path_taken) * 100
    print(f'Efficiency: {efficiency:.1f}%')
else:
    print('❌ Robot failed to reach the target.')

## 7. Animate the Path

In [ ]:
# Animate the robot's navigation
if frames:
    fig, ax = plt.subplots(figsize=(6, 6))

    colors = ['#FFFFFF', '#DDDDDD', '#000000', '#FF0000', '#00FF00']
    cmap = plt.matplotlib.colors.ListedColormap(colors)
    bounds = [0.0, 0.25, 0.75, 1.5, 2.5, 3.5]
    norm = plt.matplotlib.colors.BoundaryNorm(bounds, cmap.N)

    def update(i):
        frame_data = frames[i]
        ax.clear()
        ax.imshow(frame_data, cmap=cmap, norm=norm, origin='upper',
                  extent=[0, frame_data.shape[1], frame_data.shape[0], 0])
        ax.set_xticks(np.arange(frame_data.shape[1] + 1) - 0.5, minor=True)
        ax.set_yticks(np.arange(frame_data.shape[0] + 1) - 0.5, minor=True)
        ax.grid(which='minor', color='black', linestyle='-', linewidth=1)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f'Step {i} / {len(frames)-1}')

    ani = animation.FuncAnimation(fig, update, frames=range(len(frames)), repeat=False, interval=300)
    
    # Display inline in notebook
    plt.close()
    HTML(ani.to_jshtml())

## 8. Save Animation as GIF

In [ ]:
# Save the animation as a GIF for the README
if frames:
    save_path = '../examples/robot_animation.gif'
    ani.save(save_path, writer='pillow', fps=2)
    print(f'✅ Animation saved to {save_path}')
else:
    print('❌ No frames to save.')

---
## Summary

| | Value |
|---|---|
| Model | ResNet CNN |
| Grid Size | 20x20 |
| Actions | 8 (UP, DOWN, LEFT, RIGHT + diagonals) |
| Training | BFS-supervised imitation learning |

---
*GridNav-AI — Built with PyTorch 🔥*